# RNNTagger

If you need a POS-tagger or lemmatizer but none is available as a Python library, you may be able to loop in an external program. This demo notebook shows how output from the command-line tool [RNNTagger](https://www.cis.uni-muenchen.de/~schmid/tools/RNNTagger/) may be read in as corpus metadata. Its execution below follows Linux/macOS syntax; you will have to tweak it to fit Windows, or your particular directory structure.

## The Usual

We'll start by cloning ECHOE (as in [the `fetch_repo.ipynb` notebook](https://github.com/langeslag/ehtc/blob/main/templates/fetch_repo.ipynb)), accessing its text nodes using `lxml.etree` (as in [the demo notebook by that name](https://github.com/langeslag/ehtc/blob/main/demo/lxml.etree.ipynb)), and normalizing the token data ([ditto](https://github.com/langeslag/ehtc/blob/main/demo/lxml.etree.ipynb)). You may as well skip to the next heading.

In [1]:
from pathlib import Path
from lxml import etree
from git import Repo
import subprocess

We'll ascertain the ECHOE repository has been cloned so we have XML documents to work with:

In [2]:
# HTTPS clone point:
remote = 'https://github.com/ECHOEProject/echoe.git'
# Desired target folder name:
local = Path.cwd().parent / 'corpora' / 'echoe'
# Only clone if the target folder doesn't already exist:
if not(local.exists()):
    repo = Repo.clone_from(remote, local)
# Else, just update the working copy from remote:
else:
    repo = Repo(local)
    assert isinstance(repo, Repo)
    repo.remotes.origin.pull()
assert not repo.bare

For demonstration purposes, we'll load just a single document to minimize processor load:

In [3]:
file_id = '344.05'
filename = file_id + '.xml'
parser = etree.XMLParser(remove_blank_text=True,resolve_entities=True)
filepath = local / 'xml' / filename
tree = etree.parse(filepath, parser=parser)
root = tree.getroot()
text = root.find('.//{http://www.tei-c.org/ns/1.0}text')

In [4]:
# We'll make a list of elements to get rid of:
discard = ['abbr', 'am', 'sic', 'del', 'note', 'surplus']
# Now we define their text nodes as empty strings:
query = ['{http://www.tei-c.org/ns/1.0}' + i for i in discard]
for hit in text.iter(query):
    hit.text = ''

In [5]:
# Create a dict with characters we want replaced:
substitutions = {
    'ę': 'æ',
    'ƿ': 'w',
    'ẏ': 'y',
    'ſ': 's',
    '': 's', # Using the glyph for descending s, instead of the unicode key point
    'v': 'u',
    'j': 'i',
    '⁊': 'and',
    ' ': '',
    '\n': ''
}

# Write a function carrying out the desired operations:
def normalize(token):
    # Lowercase:
    token = token.lower()
    for k,v in substitutions.items():
        # Carry out replacements:
        token = token.replace(k, v)
    return token

In [6]:
tokens = []
for token in text.iter('{http://www.tei-c.org/ns/1.0}w'):
    # If a word element is marked as the last part of a word, add its text content to the preceding token:
    if token.get('part') == 'F':
        position = len(tokens)-1
        tokens[position] = tokens[position] + normalize(etree.tostring(token, method='text', encoding='unicode'))
    else:
        tokens.append(normalize(etree.tostring(token, method='text', encoding='unicode')))

RNNTagger expects a plaintext file, so let's output our list of tokens to a text file:

In [7]:
plaintext_folder = Path.cwd() / 'corpus_files'
plaintext_folder.mkdir(exist_ok=True)
plaintext_filename = file_id + '.txt'
plaintext_path = str(Path(plaintext_folder / plaintext_filename))
with open(plaintext_path, 'w') as f:
    f.write(' '.join(tokens))

## Running RNNTagger and Processing Its Output

We will use the stock library `subprocess` to run our command-line tagger. RNNTagger insists on being run from its own main folder, so we'll set the working directory as needed using the `cwd` argument:

In [8]:
rnntagger = subprocess.Popen(['./cmd/rnn-tagger-old-english.sh', plaintext_path], cwd='/opt/RNNTagger', stdout=subprocess.PIPE, stderr=subprocess.PIPE)
rnntagger_out, rnntagger_err = rnntagger.communicate()
output = rnntagger_out.splitlines()


Since the subprocess data retrieved is formatted as a binary string, we'll use `str.decode()` to read it in:

In [9]:
output[0].decode()

'urum\tPRO$^D'

Now it's just a matter of using that tab to separate out our data types. If you've run RNNTagger on a language for which it has a lemmatizer, you'll have two tabs, and three pieces of data per line, but for Old English we're looking at just the one tab and two units of data:

In [10]:
tagged_doc = []
for line in output:
    data = dict()
    if '\t' in line.decode():
        rubble = line.decode().split('\t')
        data['form'] = rubble[0]
        data['pos'] = rubble[1]
        tagged_doc.append(data)

In [11]:
tagged_doc

[{'form': 'urum', 'pos': 'PRO$^D'},
 {'form': 'wealdende', 'pos': 'N^D'},
 {'form': 'rihtgelyfendum', 'pos': 'ADJ^D'},
 {'form': 'a', 'pos': 'P'},
 {'form': 'worulda', 'pos': 'N^G'},
 {'form': 'woruld', 'pos': 'N^A'},
 {'form': 'minum', 'pos': 'PRO$^D'},
 {'form': 'þam', 'pos': 'D^D'},
 {'form': 'leofestan', 'pos': 'ADJS^D'},
 {'form': 'hlaforde', 'pos': 'N^D'},
 {'form': 'ofer', 'pos': 'P'},
 {'form': 'ealle', 'pos': 'Q^A'},
 {'form': 'oðre', 'pos': 'ADJ^A'},
 {'form': 'men', 'pos': 'N^A'},
 {'form': 'eorðlice', 'pos': 'ADJ^N'},
 {'form': 'kyningas', 'pos': 'N^N'},
 {'form': 'alfwold', 'pos': 'N^N'},
 {'form': 'eastengla', 'pos': 'N^G'},
 {'form': 'kyning', 'pos': 'N^N'},
 {'form': 'mid', 'pos': 'P'},
 {'form': 'rihte', 'pos': 'N^D'},
 {'form': 'and', 'pos': 'CONJ'},
 {'form': 'mid', 'pos': 'P'},
 {'form': 'gerisenum', 'pos': 'ADJ^D'},
 {'form': 'rice', 'pos': 'N^D'},
 {'form': 'healdend', 'pos': 'ADJ^N'},
 {'form': 'felix', 'pos': 'FW'},
 {'form': 'þone', 'pos': 'D^A'},
 {'form': 'ri